# 📐 The Calculus of Errors in Option Trading
## A University-Level Quantitative Finance Laboratory
### NYSE Equity Options — 5-Second Interval Data

---

> *"In mathematics the art of proposing a question must be held of higher value than solving it."*  
> — Georg Cantor

---

## Laboratory Overview

This lab investigates a single, powerful question: **how do small errors compound into large losses?**

In options trading, three primary error sources interact continuously:

1. **Volatility Estimation Error** — The implied volatility $\hat{\sigma}$ is never the true volatility $\sigma$. The difference $\epsilon_\sigma = \hat{\sigma} - \sigma$ propagates through every Greek.
2. **Numerical Approximation Error** — When we compute Greeks via finite differences, we introduce truncation error bounded by $O(h^2)$ for central differences or $O(h)$ for forward differences.
3. **Market Microstructure Friction** — The bid-ask spread creates a minimum cost floor: every round-trip trade costs at least $\frac{1}{2}(\text{ask} - \text{bid})$ per side.

### The Taylor Series Connection

The entire Black-Scholes Greek framework is a **Taylor Series expansion** of the option price $V$ with respect to its inputs. For small changes $\delta S$ in the underlying price:

$$V(S + \delta S) = V(S) + \underbrace{\frac{\partial V}{\partial S}}_{\Delta} \delta S + \frac{1}{2}\underbrace{\frac{\partial^2 V}{\partial S^2}}_{\Gamma} (\delta S)^2 + \frac{1}{6}\frac{\partial^3 V}{\partial S^3}(\delta S)^3 + \cdots$$

When we **truncate** this series after the $\Gamma$ term (as most hedging models do), we incur a **truncation error**:

$$\epsilon_{\text{trunc}} = \frac{1}{6}\frac{\partial^3 V}{\partial S^3}(\delta S)^3 + O((\delta S)^4)$$

This third-order term is called **Speed** ($\frac{\partial \Gamma}{\partial S}$) and becomes significant during large moves or near expiration.

### The Finite Difference Approximation

When computing Delta numerically with step size $h$:

**Forward Difference:** $\Delta_h = \frac{V(S+h) - V(S)}{h}$, Error $= O(h)$

**Central Difference:** $\Delta_h = \frac{V(S+h) - V(S-h)}{2h}$, Error $= O(h^2)$

The relative error between analytical and numerical Greeks is:

$$\epsilon_{\text{rel}} = \frac{|\Delta_{\text{analytical}} - \Delta_{h}|}{|\Delta_{\text{analytical}}|}$$

As $h \to 0$: truncation error $\to 0$, but **round-off error $\to \infty$** (floating point precision). The optimal $h$ balances these two error sources.

---

**Learning Objectives:**
- Implement and compare analytical vs. numerical Greeks
- Quantify error propagation from volatility misestimation
- Build and backtest delta-neutral and volatility arbitrage strategies
- Analyze the computational cost-accuracy tradeoff

**Estimated Lab Duration:** 80 minutes  
**Prerequisites:** Calculus II, Probability Theory, Python fundamentals

---
## Section 0: Environment Setup

Run the cell below once. QuantLib requires a C++ backend — on Google Colab or restricted environments, uncomment the conda line.

In [5]:
# ============================================================
# SECTION 0: ENVIRONMENT SETUP
# Run once. Restart kernel after installation if needed.
# ============================================================

import subprocess, sys

def pip_install(*packages):
    """Install packages with progress feedback."""
    for pkg in packages:
        print(f"  Installing {pkg}...", end=" ")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg, "-q"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print("✓")
        else:
            print(f"⚠ Warning: {result.stderr[:80]}")

print("Installing core dependencies...")
pip_install(
    "QuantLib-Python",      # Analytical option pricing & Greeks
    "scipy",                # Numerical finite-difference Greeks
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
)

print("\nInstalling options-specific libraries...")
pip_install(
    "optionsflow",          # OptionsFlow market data utilities
    "optionstratlib",       # OptionStratLib strategy construction
    "greeks",               # greeks-package for fast Greek calculation
)

print("\n✅ All packages installed. Proceed to the next cell.")
print("   NOTE: If QuantLib install failed, try: conda install -c conda-forge quantlib-python")

Installing core dependencies...
  Installing QuantLib-Python... ✓
  Installing scipy... ✓
  Installing numpy... ✓
  Installing pandas... ✓
  Installing matplotlib... ✓
  Installing seaborn... ✓

Installing options-specific libraries...
  Installing optionsflow... ⚠ Warning: ERROR: Could not find a version that satisfies the requirement optionsflow 
  Installing optionstratlib... ⚠ Warning: ERROR: Could not find a version that satisfies the requirement optionstratl
  Installing greeks... ⚠ Warning:   error: subprocess-exited-with-error
  
  × Bu

✅ All packages installed. Proceed to the next cell.
   NOTE: If QuantLib install failed, try: conda install -c conda-forge quantlib-python


In [6]:
# ============================================================
# CORE IMPORTS
# ============================================================

import warnings
warnings.filterwarnings('ignore')

import time
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.stats import norm
from scipy.optimize import brentq
from typing import Optional, Dict, Tuple, List
from dataclasses import dataclass, field
from collections import deque

# QuantLib
try:
    import QuantLib as ql
    QUANTLIB_AVAILABLE = True
    print("✓ QuantLib version:", ql.__version__)
except ImportError:
    QUANTLIB_AVAILABLE = False
    print("⚠ QuantLib not available — falling back to pure Black-Scholes implementation")

# greeks-package
try:
    import greeks
    GREEKS_AVAILABLE = True
    print("✓ greeks package loaded")
except ImportError:
    GREEKS_AVAILABLE = False
    print("⚠ greeks package not available — using internal implementation")

# OptionStratLib
try:
    import optionstratlib as osl
    OPTIONSTRATLIB_AVAILABLE = True
    print("✓ optionstratlib loaded")
except ImportError:
    OPTIONSTRATLIB_AVAILABLE = False
    print("⚠ optionstratlib not available — using internal vol-arb implementation")

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

SEED = 42
np.random.seed(SEED)
print("\n✅ Environment ready.")

✓ QuantLib version: 1.41
⚠ greeks package not available — using internal implementation
⚠ optionstratlib not available — using internal vol-arb implementation

✅ Environment ready.


---
## Section 1: Data Generation & The `NYSE_Streamer` API Simulator

Real market data is tick-by-tick. We simulate a **5-second interval dataset** of NYSE equity options.  
The `NYSE_Streamer` enforces **no-look-ahead bias** — each call to `.step()` reveals only the current bar; the future is invisible.

> **Why does look-ahead bias matter?**  
> If your strategy can 'see' tomorrow's volatility today, its backtest is meaningless. The `NYSE_Streamer` enforces the **information frontier**: you only know what you would have known at time $t$.

In [7]:
# ============================================================
# SECTION 1A: SYNTHETIC MARKET DATA GENERATOR
# Generates a realistic 5-second interval options dataset
# ============================================================

def generate_synthetic_options_data(
    n_bars: int = 4680,          # ~6.5 hours of 5-sec bars (one full trading day)
    S0: float = 185.0,           # Initial underlying price (e.g. AAPL-like)
    K: float = 185.0,            # Strike price (ATM)
    T_days: float = 30,          # Days to expiration at start
    r: float = 0.053,            # Risk-free rate (Fed Funds ~5.3%)
    sigma0: float = 0.22,        # Initial implied volatility
    option_type: str = 'call',
    seed: int = 42
) -> pd.DataFrame:
    """
    Generate synthetic NYSE-style options data at 5-second intervals.
    
    The underlying follows Geometric Brownian Motion with stochastic
    volatility (Ornstein-Uhlenbeck mean-reverting vol process).
    """
    rng = np.random.default_rng(seed)
    dt = 5 / (252 * 6.5 * 3600)  # 5 seconds in years
    
    # ---- Underlying price: GBM ----
    dW = rng.standard_normal(n_bars) * np.sqrt(dt)
    
    # Stochastic volatility (OU process)
    kappa, theta, xi = 2.0, sigma0, 0.4   # mean-rev speed, long-run vol, vol-of-vol
    sigma = np.zeros(n_bars)
    sigma[0] = sigma0
    dZ = rng.standard_normal(n_bars) * np.sqrt(dt)
    rho_sv = -0.65   # leverage effect: price up -> vol down
    
    for i in range(1, n_bars):
        sigma[i] = np.abs(
            sigma[i-1] + kappa * (theta - sigma[i-1]) * dt
            + xi * sigma[i-1] * (rho_sv * dW[i-1] + np.sqrt(1 - rho_sv**2) * dZ[i-1])
        )
    sigma = np.clip(sigma, 0.05, 0.95)
    
    # Underlying price path
    uPrice = np.zeros(n_bars)
    uPrice[0] = S0
    for i in range(1, n_bars):
        uPrice[i] = uPrice[i-1] * np.exp(
            (r - 0.5 * sigma[i-1]**2) * dt + sigma[i-1] * dW[i-1]
        )
    
    # Time to expiration (decreasing)
    T_arr = np.linspace(T_days / 252, 0.5 / 252, n_bars)  # expires intraday
    
    # ---- Black-Scholes pricing ----
    def bs_price_greeks(S, K, T, r, sig, opt_type='call'):
        if T < 1e-9:
            intrinsic = max(S - K, 0) if opt_type == 'call' else max(K - S, 0)
            return intrinsic, 1.0 if S > K else 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
        d1 = (np.log(S / K) + (r + 0.5 * sig**2) * T) / (sig * np.sqrt(T))
        d2 = d1 - sig * np.sqrt(T)
        pdf_d1 = norm.pdf(d1)
        if opt_type == 'call':
            price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
            delta = norm.cdf(d1)
        else:
            price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
            delta = norm.cdf(d1) - 1
        gamma = pdf_d1 / (S * sig * np.sqrt(T))
        vega  = S * pdf_d1 * np.sqrt(T) / 100          # per 1% vol change
        theta_val = (-(S * pdf_d1 * sig) / (2 * np.sqrt(T))
                     - r * K * np.exp(-r * T) * norm.cdf(d2 if opt_type=='call' else -d2)) / 365
        rho_val = (K * T * np.exp(-r * T) *
                   norm.cdf(d2 if opt_type=='call' else -d2)) / 100
        return price, delta, gamma, vega, theta_val, rho_val, d1
    
    rows = []
    for i in range(n_bars):
        S, sig, T = uPrice[i], sigma[i], T_arr[i]
        
        # Implied vol has noise vs realized vol (this IS the error we study)
        iv_noise = rng.normal(0, 0.008)   # ~0.8% IV estimation error
        iv = np.clip(sig + iv_noise, 0.05, 0.90)
        
        price, delta, gamma, vega, theta_v, rho_v, d1 = bs_price_greeks(S, K, T, r, iv, option_type)
        
        # Market microstructure: bid-ask spread
        half_spread = max(0.01, price * 0.005 + 0.02 * rng.exponential())
        bid = max(0.01, price - half_spread)
        ask = price + half_spread
        last_noise = rng.uniform(-half_spread * 0.5, half_spread * 0.5)
        last = np.clip(price + last_noise, bid, ask)
        
        bid_size = int(rng.integers(1, 50) * 100)
        ask_size = int(rng.integers(1, 50) * 100)
        
        timestamp = pd.Timestamp('2024-01-15 09:30:00') + pd.Timedelta(seconds=i * 5)
        
        rows.append({
            'timestamp': timestamp,
            'uPrice': round(S, 4),
            'volatility': round(iv, 6),
            'realized_vol': round(sig, 6),   # Hidden from students in real use!
            'time_to_expiry': round(T, 8),
            '0_last': round(last, 4),
            '0_bid': round(bid, 4),
            '0_ask': round(ask, 4),
            '0_bidSize': bid_size,
            '0_askSize': ask_size,
            '0_delta': round(delta, 6),
            '0_gamma': round(gamma, 6),
            '0_vega': round(vega, 6),
            '0_theta': round(theta_v, 6),
            '0_rho': round(rho_v, 6),
            'strike': K,
            'option_type': option_type,
            'risk_free_rate': r,
        })
    
    df = pd.DataFrame(rows).set_index('timestamp')
    print(f"✅ Generated {n_bars} bars | S: {uPrice[0]:.2f}→{uPrice[-1]:.2f} "
          f"| Vol: {sigma.mean():.1%}±{sigma.std():.1%} "
          f"| Avg spread: ${(df['0_ask']-df['0_bid']).mean():.3f}")
    return df


# Generate the primary dataset
df_market = generate_synthetic_options_data(n_bars=4680, seed=SEED)
df_market.head(3)

✅ Generated 4680 bars | S: 185.00→182.39 | Vol: 22.4%±0.3% | Avg spread: $0.072


,uPrice,volatility,realized_vol,time_to_expiry,0_last,0_bid,0_ask,0_bidSize,0_askSize,0_delta,0_gamma,0_vega,0_theta,0_rho,strike,option_type,risk_free_rate
timestamp,,,,,,,,,,,,,,,,,
2024-01-15 09:30:00,185.0000,0.217156,0.220000,0.119048,6.0797,6.0262,6.1988,3000,400,0.548421,0.028569,0.252771,-0.077007,0.113506,185.0,call,0.053
2024-01-15 09:30:05,185.0114,0.203534,0.220083,0.119023,5.7653,5.7429,5.8048,2600,4100,0.550065,0.030466,0.252630,-0.073118,0.114255,185.0,call,0.053
2024-01-15 09:30:10,184.9724,0.214690,0.220144,0.118998,6.0228,5.9714,6.0959,1000,2600,0.547829,0.028913,0.252726,-0.076298,0.113404,185.0,call,0.053


In [8]:
# ============================================================
# SECTION 1B: NYSE_Streamer — The Zero-Look-Ahead API Simulator
# ============================================================

@dataclass
class OrderRecord:
    order_id: int
    timestamp: pd.Timestamp
    side: str              # 'buy' or 'sell'
    quantity: int          # number of contracts (each = 100 shares)
    order_type: str        # 'market', 'limit'
    limit_price: Optional[float]
    fill_price: Optional[float] = None
    status: str = 'pending'   # pending, filled, rejected, cancelled
    asset: str = 'option'     # 'option' or 'underlying'


class NYSE_Streamer:
    """
    A zero-look-ahead market data simulator for NYSE equity options.
    
    Design Principles:
    ------------------
    1. NO LOOK-AHEAD: The internal cursor `_idx` is strictly monotone-increasing.
       Accessing future data raises IndexError.
    2. REALISTIC FILLS: Market orders fill at the ask (buy) or bid (sell).
       Limit orders only fill if the limit price crosses the market.
    3. ORDER BOOK CONSTRAINTS: Orders larger than available size are rejected.
    4. TRANSACTION COSTS: Each fill incurs a configurable commission.
    
    Usage:
    ------
        streamer = NYSE_Streamer(df_market)
        state = streamer.step()              # advance one bar
        streamer.submit_order('buy', 1)      # market order, 1 contract
    """
    
    def __init__(
        self,
        data: pd.DataFrame,
        commission_per_contract: float = 0.65,
        multiplier: int = 100,              # standard US equity option multiplier
        initial_cash: float = 100_000.0,
    ):
        # Defensive copy — students cannot mutate the source data
        self._data = data.copy(deep=True)
        self._n = len(self._data)
        self._idx = -1           # Not yet started; call step() first
        self._started = False
        
        self.commission = commission_per_contract
        self.multiplier = multiplier
        
        # Portfolio state
        self.cash = initial_cash
        self.option_position = 0     # contracts held (+ long, - short)
        self.stock_position = 0      # shares of underlying
        self.orders: List[OrderRecord] = []
        self._order_counter = 0
        
        # PnL tracking
        self._pnl_history: List[float] = []
        self._nav_history: List[float] = []
        self._decision_times: List[float] = []   # seconds per decision
        
        # Internal: last fill prices for MTM
        self._option_avg_cost: float = 0.0
        self._stock_avg_cost: float = 0.0
    
    # ------------------------------------------------------------------
    # CORE API
    # ------------------------------------------------------------------
    
    def step(self) -> Dict:
        """
        Advance the stream by one 5-second bar.
        
        Returns:
            A dict of CURRENT market state. Future bars are NOT accessible.
        
        Raises:
            StopIteration: When the stream is exhausted.
        """
        self._idx += 1
        if self._idx >= self._n:
            raise StopIteration("Stream exhausted — end of trading day.")
        
        self._started = True
        row = self._data.iloc[self._idx]
        
        # Process any pending limit orders at this bar's prices
        self._process_pending_orders(row)
        
        # Record NAV
        nav = self._compute_nav(row)
        self._nav_history.append(nav)
        
        # Expose ONLY current-bar information
        state = {
            'bar_index': self._idx,
            'timestamp': row.name,
            'uPrice': row['uPrice'],
            'volatility': row['volatility'],
            'time_to_expiry': row['time_to_expiry'],
            '0_last': row['0_last'],
            '0_bid': row['0_bid'],
            '0_ask': row['0_ask'],
            '0_bidSize': row['0_bidSize'],
            '0_askSize': row['0_askSize'],
            '0_delta': row['0_delta'],
            '0_gamma': row['0_gamma'],
            '0_vega': row['0_vega'],
            '0_theta': row['0_theta'],
            '0_rho': row['0_rho'],
            'strike': row['strike'],
            'option_type': row['option_type'],
            'risk_free_rate': row['risk_free_rate'],
            # Portfolio info
            'cash': self.cash,
            'option_position': self.option_position,
            'stock_position': self.stock_position,
            'nav': nav,
            'bars_remaining': self._n - self._idx - 1,
        }
        return state
    
    def submit_order(
        self,
        side: str,
        quantity: int,
        order_type: str = 'market',
        limit_price: Optional[float] = None,
        asset: str = 'option',
        decision_time_sec: float = 0.0,
    ) -> OrderRecord:
        """
        Submit a trading order.
        
        Parameters:
        -----------
        side         : 'buy' or 'sell'
        quantity     : Number of contracts (options) or shares (underlying)
        order_type   : 'market' or 'limit'
        limit_price  : Required for limit orders
        asset        : 'option' or 'underlying'
        decision_time_sec: Wall-clock time taken for this decision (for efficiency scoring)
        
        Returns:
        --------
        OrderRecord with fill details
        """
        if not self._started:
            raise RuntimeError("Call step() before submitting orders.")
        if side not in ('buy', 'sell'):
            raise ValueError(f"Invalid side: {side}. Must be 'buy' or 'sell'.")
        if quantity <= 0:
            raise ValueError("Quantity must be positive.")
        if order_type == 'limit' and limit_price is None:
            raise ValueError("limit_price required for limit orders.")
        
        self._order_counter += 1
        row = self._data.iloc[self._idx]
        
        order = OrderRecord(
            order_id=self._order_counter,
            timestamp=row.name,
            side=side,
            quantity=quantity,
            order_type=order_type,
            limit_price=limit_price,
            asset=asset,
        )
        
        if decision_time_sec > 0:
            self._decision_times.append(decision_time_sec)
        
        # Market orders fill immediately at current bar
        if order_type == 'market':
            self._fill_order(order, row)
        else:
            order.status = 'pending'
        
        self.orders.append(order)
        return order
    
    # ------------------------------------------------------------------
    # INTERNAL MECHANICS
    # ------------------------------------------------------------------
    
    def _fill_order(self, order: OrderRecord, row: pd.Series) -> None:
        """Execute an order at realistic market prices with size constraints."""
        if order.asset == 'option':
            available = row['0_askSize'] if order.side == 'buy' else row['0_bidSize']
            if order.quantity * 100 > available:
                order.status = 'rejected'
                order.fill_price = None
                return
            fill_price = row['0_ask'] if order.side == 'buy' else row['0_bid']
            notional = fill_price * order.quantity * self.multiplier
            commission = self.commission * order.quantity
            
            if order.side == 'buy':
                if self.cash < notional + commission:
                    order.status = 'rejected'
                    return
                self.cash -= (notional + commission)
                self.option_position += order.quantity
                # Update average cost
                self._option_avg_cost = fill_price
            else:
                self.cash += (notional - commission)
                self.option_position -= order.quantity
                self._option_avg_cost = fill_price
        
        elif order.asset == 'underlying':
            fill_price = row['uPrice'] * (1 + 0.0001 * (1 if order.side == 'buy' else -1))  # 1bp slip
            notional = fill_price * order.quantity
            commission = 0.005 * order.quantity  # $0.005/share
            
            if order.side == 'buy':
                if self.cash < notional + commission:
                    order.status = 'rejected'
                    return
                self.cash -= (notional + commission)
                self.stock_position += order.quantity
            else:
                self.cash += (notional - commission)
                self.stock_position -= order.quantity
        
        order.fill_price = fill_price
        order.status = 'filled'
    
    def _process_pending_orders(self, row: pd.Series) -> None:
        """Check pending limit orders against current prices."""
        for order in self.orders:
            if order.status != 'pending':
                continue
            if order.asset == 'option':
                if order.side == 'buy' and order.limit_price >= row['0_ask']:
                    self._fill_order(order, row)
                elif order.side == 'sell' and order.limit_price <= row['0_bid']:
                    self._fill_order(order, row)
    
    def _compute_nav(self, row: pd.Series) -> float:
        """Mark-to-market NAV."""
        option_mtm = self.option_position * row['0_last'] * self.multiplier
        stock_mtm  = self.stock_position * row['uPrice']
        return self.cash + option_mtm + stock_mtm
    
    # ------------------------------------------------------------------
    # ANALYTICS METHODS
    # ------------------------------------------------------------------
    
    def get_performance_metrics(self, risk_free_rate: float = 0.053) -> Dict:
        """Compute final performance metrics for the 80-minute challenge."""
        if len(self._nav_history) < 2:
            return {}
        
        navs = np.array(self._nav_history)
        initial_nav = navs[0]
        returns = np.diff(navs) / navs[:-1]
        
        total_pnl = navs[-1] - initial_nav
        
        # Annualized Sharpe (bars are 5-sec; 252*6.5*3600/5 bars/year)
        bars_per_year = 252 * 6.5 * 3600 / 5
        ann_factor = np.sqrt(bars_per_year)
        rf_per_bar = risk_free_rate / bars_per_year
        excess_ret = returns - rf_per_bar
        sharpe = (excess_ret.mean() / (excess_ret.std() + 1e-12)) * ann_factor
        
        # Max drawdown
        peak = np.maximum.accumulate(navs)
        drawdown = (navs - peak) / peak
        max_dd = drawdown.min()
        
        # Computational efficiency
        avg_decision_ms = np.mean(self._decision_times) * 1000 if self._decision_times else np.nan
        efficiency_score = max(0, 100 - avg_decision_ms * 10)   # lose 10pts per extra ms
        
        return {
            'Total PnL ($)': round(total_pnl, 2),
            'Return (%)': round(100 * total_pnl / initial_nav, 3),
            'Sharpe Ratio (Ann.)': round(sharpe, 4),
            'Max Drawdown (%)': round(100 * max_dd, 3),
            'N Trades': sum(1 for o in self.orders if o.status == 'filled'),
            'N Rejected': sum(1 for o in self.orders if o.status == 'rejected'),
            'Avg Decision Time (ms)': round(avg_decision_ms, 3),
            'Computational Efficiency Score': round(efficiency_score, 1),
        }
    
    def reset(self, initial_cash: float = 100_000.0) -> None:
        """Reset the streamer to bar 0 for re-running experiments."""
        self._idx = -1
        self._started = False
        self.cash = initial_cash
        self.option_position = 0
        self.stock_position = 0
        self.orders.clear()
        self._order_counter = 0
        self._nav_history.clear()
        self._pnl_history.clear()
        self._decision_times.clear()
        self._option_avg_cost = 0.0
        self._stock_avg_cost = 0.0


# --- Quick API demo ---
streamer = NYSE_Streamer(df_market)
state = streamer.step()
print("First bar state:")
for k, v in state.items():
    if not isinstance(v, (str, pd.Timestamp, int)) or k in ('bar_index', 'option_position'):
        print(f"  {k:25s}: {v}")

First bar state:
  bar_index                : 0
  uPrice                   : 185.0
  volatility               : 0.217156
  time_to_expiry           : 0.11904762
  0_last                   : 6.0797
  0_bid                    : 6.0262
  0_ask                    : 6.1988
  0_bidSize                : 3000
  0_askSize                : 400
  0_delta                  : 0.548421
  0_gamma                  : 0.028569
  0_vega                   : 0.252771
  0_theta                  : -0.077007
  0_rho                    : 0.113506
  strike                   : 185.0
  risk_free_rate           : 0.053
  cash                     : 100000.0
  option_position          : 0
  nav                      : 100000.0


---
## Section 2: Computational Mathematics — The Error Analysis Module

### 2.1 The Three-Source Error Framework

Let $V(S, \sigma, t)$ be the true option value. Any computed approximation $\hat{V}$ deviates by:

$$\hat{V} - V = \underbrace{\frac{\partial V}{\partial \sigma} \epsilon_\sigma}_{\text{Vol Error}} + \underbrace{\frac{h^2}{6} \frac{\partial^3 V}{\partial S^3}}_{\text{Truncation Error}} + \underbrace{\frac{\epsilon_{\text{machine}}}{h}}_{\text{Round-off Error}}$$

where $\epsilon_{\sigma} = \hat{\sigma} - \sigma$ is the vol estimation error and $h$ is the finite-difference step.

### 2.2 Optimal Step Size

The total numerical error in computing $\Delta$ via central differences is minimized when:

$$h^* = \left(\frac{3 \epsilon_{\text{machine}}}{|\Delta'''(S)|}\right)^{1/4}$$

For double-precision arithmetic, $\epsilon_{\text{machine}} \approx 2.2 \times 10^{-16}$, giving $h^* \approx 0.01$ (1% of spot price) as a typical rule of thumb.

### 2.3 The Pin Risk Surface

Near expiration ($T \to 0$) with $S \approx K$ (the strike), Gamma explodes:

$$\Gamma \to \frac{1}{S \sigma \sqrt{2\pi T}} \quad \text{as} \quad T \to 0$$

This is **pin risk** — the option's Delta discontinuously jumps between 0 and 1, making any finite-difference approximation wildly inaccurate. We will visualize this as an 'error surface.'

In [9]:
# ============================================================
# SECTION 2A: BLACK-SCHOLES ENGINE
# Analytical (QuantLib + pure Python fallback) vs. Numerical (SciPy)
# ============================================================

class BlackScholesEngine:
    """
    Dual-mode Black-Scholes engine.
    
    Modes:
    - 'analytical' : Closed-form Greeks via QuantLib (if available) or direct formulas
    - 'numerical'  : Finite-difference Greeks via SciPy-style central differences
    """
    
    def __init__(self, use_quantlib: bool = True):
        self.use_quantlib = use_quantlib and QUANTLIB_AVAILABLE
        if self.use_quantlib:
            print("✓ BSEngine using QuantLib analytical mode")
        else:
            print("✓ BSEngine using direct closed-form analytical mode")
    
    # ---------- ANALYTICAL (Exact) ----------
    
    def _d1_d2(self, S, K, T, r, sigma):
        if T < 1e-10: return np.nan, np.nan
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return d1, d2
    
    def analytical_price(self, S, K, T, r, sigma, opt_type='call'):
        if T < 1e-10:
            return max(S-K, 0) if opt_type=='call' else max(K-S, 0)
        if self.use_quantlib:
            return self._ql_price(S, K, T, r, sigma, opt_type)
        d1, d2 = self._d1_d2(S, K, T, r, sigma)
        if opt_type == 'call':
            return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        else:
            return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    
    def analytical_delta(self, S, K, T, r, sigma, opt_type='call'):
        if T < 1e-10: return (1.0 if S > K else 0.0) if opt_type=='call' else (-1.0 if S < K else 0.0)
        if self.use_quantlib:
            return self._ql_greek(S, K, T, r, sigma, opt_type, 'delta')
        d1, _ = self._d1_d2(S, K, T, r, sigma)
        return norm.cdf(d1) if opt_type == 'call' else norm.cdf(d1) - 1
    
    def analytical_gamma(self, S, K, T, r, sigma, opt_type='call'):
        if T < 1e-10: return 0.0
        if self.use_quantlib:
            return self._ql_greek(S, K, T, r, sigma, opt_type, 'gamma')
        d1, _ = self._d1_d2(S, K, T, r, sigma)
        return norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    # ---------- QuantLib Backend ----------
    
    def _ql_setup(self, S, K, T, r, sigma, opt_type):
        today = ql.Date.todaysDate()
        ql.Settings.instance().evaluationDate = today
        T_days = max(1, int(round(T * 365)))
        expiry = today + T_days
        payoff = ql.PlainVanillaPayoff(
            ql.Option.Call if opt_type == 'call' else ql.Option.Put, K
        )
        exercise = ql.EuropeanExercise(expiry)
        option = ql.VanillaOption(payoff, exercise)
        spot_q = ql.SimpleQuote(S)
        vol_q  = ql.SimpleQuote(sigma)
        rf_q   = ql.SimpleQuote(r)
        div_q  = ql.SimpleQuote(0.0)
        dc = ql.Actual365Fixed()
        spot_h = ql.QuoteHandle(spot_q)
        vol_ts  = ql.BlackConstantVol(today, ql.NullCalendar(), ql.QuoteHandle(vol_q), dc)
        rf_ts   = ql.FlatForward(today, ql.QuoteHandle(rf_q), dc)
        div_ts  = ql.FlatForward(today, ql.QuoteHandle(div_q), dc)
        process = ql.BlackScholesMertonProcess(
            spot_h,
            ql.YieldTermStructureHandle(div_ts),
            ql.YieldTermStructureHandle(rf_ts),
            ql.BlackVolTermStructureHandle(vol_ts),
        )
        engine = ql.AnalyticEuropeanEngine(process)
        option.setPricingEngine(engine)
        return option
    
    def _ql_price(self, S, K, T, r, sigma, opt_type):
        try:
            return self._ql_setup(S, K, T, r, sigma, opt_type).NPV()
        except:
            return self.analytical_price(S, K, T, r, sigma, opt_type)
    
    def _ql_greek(self, S, K, T, r, sigma, opt_type, greek):
        try:
            opt = self._ql_setup(S, K, T, r, sigma, opt_type)
            return getattr(opt, greek)()
        except:
            return None
    
    # ---------- NUMERICAL (Finite Difference) ----------
    
    def numerical_delta(self, S, K, T, r, sigma, opt_type='call', h=None):
        """
        Central-difference approximation to Delta.
        Error = O(h^2) + O(eps_machine / h)
        """
        if h is None:
            h = S * 0.01   # 1% of spot (typical default)
        V_up   = self.analytical_price(S + h, K, T, r, sigma, opt_type)
        V_down = self.analytical_price(S - h, K, T, r, sigma, opt_type)
        return (V_up - V_down) / (2 * h)
    
    def numerical_gamma(self, S, K, T, r, sigma, opt_type='call', h=None):
        """
        Central-difference approximation to Gamma.
        Error = O(h^2) + O(eps_machine / h^2)
        """
        if h is None:
            h = S * 0.01
        V_up   = self.analytical_price(S + h, K, T, r, sigma, opt_type)
        V_mid  = self.analytical_price(S,     K, T, r, sigma, opt_type)
        V_down = self.analytical_price(S - h, K, T, r, sigma, opt_type)
        return (V_up - 2 * V_mid + V_down) / (h ** 2)
    
    def error_analysis(self, S, K, T, r, sigma, opt_type='call',
                       h_values=None) -> pd.DataFrame:
        """
        Compute relative & absolute errors for Delta and Gamma
        across a range of step sizes h.
        
        Returns a DataFrame suitable for plotting the 'error curve'.
        """
        if h_values is None:
            h_values = np.logspace(-7, 0, 60) * S
        
        true_delta = self.analytical_delta(S, K, T, r, sigma, opt_type)
        true_gamma = self.analytical_gamma(S, K, T, r, sigma, opt_type)
        
        rows = []
        for h in h_values:
            num_d = self.numerical_delta(S, K, T, r, sigma, opt_type, h)
            num_g = self.numerical_gamma(S, K, T, r, sigma, opt_type, h)
            rows.append({
                'h': h,
                'h_rel': h / S,
                'num_delta': num_d,
                'num_gamma': num_g,
                'delta_abs_err': abs(num_d - true_delta),
                'delta_rel_err': abs(num_d - true_delta) / (abs(true_delta) + 1e-15),
                'gamma_abs_err': abs(num_g - true_gamma),
                'gamma_rel_err': abs(num_g - true_gamma) / (abs(true_gamma) + 1e-15),
            })
        return pd.DataFrame(rows)


bs = BlackScholesEngine(use_quantlib=QUANTLIB_AVAILABLE)

# Demo: single state
row0 = df_market.iloc[100]
S, K, T = row0['uPrice'], row0['strike'], row0['time_to_expiry']
r, sigma = row0['risk_free_rate'], row0['volatility']

a_delta = bs.analytical_delta(S, K, T, r, sigma)
n_delta = bs.numerical_delta(S, K, T, r, sigma, h=S*0.01)
a_gamma = bs.analytical_gamma(S, K, T, r, sigma)
n_gamma = bs.numerical_gamma(S, K, T, r, sigma, h=S*0.01)

print(f"\n{'':>30s} {'Analytical':>12s} {'Numerical':>12s} {'Rel Error':>12s}")
print("-" * 68)
print(f"{'Delta':>30s} {a_delta:>12.6f} {n_delta:>12.6f} {abs(a_delta-n_delta)/abs(a_delta):>12.2e}")
print(f"{'Gamma':>30s} {a_gamma:>12.6f} {n_gamma:>12.6f} {abs(a_gamma-n_gamma)/abs(a_gamma):>12.2e}")

✓ BSEngine using QuantLib analytical mode

                                 Analytical    Numerical    Rel Error
--------------------------------------------------------------------
                         Delta     0.542331     0.542129     3.73e-04
                         Gamma     0.027770     0.027734     1.33e-03


In [10]:
# ============================================================
# SECTION 2B: ERROR CURVES — Delta & Gamma vs. Step Size h
# ============================================================

# ---- Student Exercise ----
# TASK: Observe where truncation error (large h) and round-off error (small h) dominate.
# The 'sweet spot' h* minimizes total error. Can you derive it analytically?

err_df = bs.error_analysis(S, K, T, r, sigma)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, greek, color in zip(axes, ['delta', 'gamma'], ['steelblue', 'darkorange']):
    ax.loglog(err_df['h_rel'], err_df[f'{greek}_rel_err'],
              color=color, lw=2, label='Central Difference Error')
    
    # Theoretical O(h^2) slope reference
    h_ref = err_df['h_rel'].values
    scale = err_df[f'{greek}_rel_err'].iloc[40] / h_ref[40]**2
    ax.loglog(h_ref, scale * h_ref**2, 'k--', lw=1, alpha=0.7, label=r'$O(h^2)$ slope')
    
    # Mark optimal h
    best_idx = err_df[f'{greek}_rel_err'].idxmin()
    best_h = err_df.loc[best_idx, 'h_rel']
    best_e = err_df.loc[best_idx, f'{greek}_rel_err']
    ax.axvline(best_h, color='red', linestyle=':', alpha=0.8, lw=1.5)
    ax.scatter([best_h], [best_e], color='red', zorder=5, s=80,
               label=f'Optimal h* ≈ {best_h:.1e}')
    
    ax.set_xlabel('Relative Step Size  $h/S$', fontsize=12)
    ax.set_ylabel('Relative Error', fontsize=12)
    ax.set_title(f'{greek.capitalize()} Numerical Error vs. Step Size\n'
                 f'(S={S:.1f}, K={K:.1f}, T={T*365:.1f}d, σ={sigma:.1%})', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_ylim([1e-16, 1e2])
    ax.text(0.05, 0.95, 'Round-off\nDominates', transform=ax.transAxes,
            va='top', ha='left', fontsize=8, color='purple',
            bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.7))
    ax.text(0.75, 0.95, 'Truncation\nDominates', transform=ax.transAxes,
            va='top', ha='left', fontsize=8, color='saddlebrown',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.suptitle('The Calculus of Numerical Errors — Central Difference Greeks', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('error_curves.png', dpi=130, bbox_inches='tight')
plt.show()
print("\n📌 Key Insight: Error is NOT monotone in h. There is an optimal step size.")
print(f"   Optimal h/S for Delta ≈ {err_df.loc[err_df['delta_rel_err'].idxmin(), 'h_rel']:.2e}")
print(f"   Optimal h/S for Gamma ≈ {err_df.loc[err_df['gamma_rel_err'].idxmin(), 'h_rel']:.2e}")

RecursionError: maximum recursion depth exceeded

In [ ]:
# ============================================================
# SECTION 2C: PIN RISK — The Error Surface Near Expiration
# ============================================================
# This surface shows WHERE the Black-Scholes model (and numerical Greeks)
# catastrophically breaks down: near-the-money, near-expiration (Pin Risk).

T_grid = np.linspace(0.5/365, 25/365, 60)    # 0.5 days to 25 days
M_grid = np.linspace(0.85, 1.15, 60)          # Moneyness S/K
TT, MM = np.meshgrid(T_grid, M_grid)

h_fixed = 0.01 * K   # fixed step for numerical Gamma

SIGMA_SURF = 0.22
R_SURF = 0.053

gamma_analytical = np.zeros_like(TT)
gamma_numerical  = np.zeros_like(TT)
relative_error   = np.zeros_like(TT)

for i in range(TT.shape[0]):
    for j in range(TT.shape[1]):
        t = TT[i, j]
        s = MM[i, j] * K
        g_a = bs.analytical_gamma(s, K, t, R_SURF, SIGMA_SURF)
        g_n = bs.numerical_gamma(s, K, t, R_SURF, SIGMA_SURF, h=h_fixed)
        gamma_analytical[i, j] = g_a
        gamma_numerical[i, j]  = g_n
        relative_error[i, j]   = abs(g_n - g_a) / (abs(g_a) + 1e-10)

# Clip for visibility
err_plot = np.clip(np.log10(relative_error + 1e-10), -6, 2)
gamma_plot = np.clip(gamma_analytical, 0, 2.0)

fig = plt.figure(figsize=(18, 7))

# --- Panel 1: Gamma Surface ---
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(MM, TT * 365, gamma_plot, cmap='plasma', alpha=0.85)
ax1.set_xlabel('Moneyness (S/K)', labelpad=8)
ax1.set_ylabel('Days to Expiry', labelpad=8)
ax1.set_zlabel('Gamma', labelpad=8)
ax1.set_title('Gamma Surface\n(σ=22%, clipped at 2.0)', fontsize=11)
fig.colorbar(surf1, ax=ax1, shrink=0.5, label='Gamma')
ax1.view_init(elev=25, azim=-60)

# --- Panel 2: Numerical Error Surface ---
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(MM, TT * 365, err_plot, cmap='RdYlGn_r', alpha=0.85)
ax2.set_xlabel('Moneyness (S/K)', labelpad=8)
ax2.set_ylabel('Days to Expiry', labelpad=8)
ax2.set_zlabel('log₁₀(Rel Error)', labelpad=8)
ax2.set_title(f'Gamma Numerical Error Surface\n(h={h_fixed:.2f}, log scale) — PIN RISK ZONE', fontsize=11)
fig.colorbar(surf2, ax=ax2, shrink=0.5, label='log₁₀(Rel Error)')
ax2.view_init(elev=25, azim=-60)

plt.suptitle('Pin Risk: Where Black-Scholes Numerical Greeks Break Down',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pin_risk_surface.png', dpi=130, bbox_inches='tight')
plt.show()

print("📌 The 'spike' at S/K=1.0 and T→0 is the PIN RISK zone.")
print("   Here, Gamma → ∞ and ANY numerical approximation fails completely.")
print("   Real traders manage this by reducing position size or closing early.")

In [ ]:
# ============================================================
# SECTION 2D: VOLATILITY ESTIMATION ERROR PROPAGATION
# ============================================================
# Task: Show how a small vol error epsilon_sigma creates a
# cascade of Greek errors via the Taylor expansion.

S_base, K_base = 185.0, 185.0
T_base = 21 / 365   # 21 days
r_base = 0.053
true_sigma = 0.22

# Vol estimation errors from -5% to +5%
epsilon_vol = np.linspace(-0.05, 0.05, 200)
sigma_hat = true_sigma + epsilon_vol

true_price = bs.analytical_price(S_base, K_base, T_base, r_base, true_sigma)
true_delta_v = bs.analytical_delta(S_base, K_base, T_base, r_base, true_sigma)
true_gamma_v = bs.analytical_gamma(S_base, K_base, T_base, r_base, true_sigma)

# First-order approximation: dV ≈ Vega * epsilon_sigma
# (Vega = dV/d_sigma)
h_sig = true_sigma * 0.001
vega_numerical = (bs.analytical_price(S_base, K_base, T_base, r_base, true_sigma + h_sig) -
                  bs.analytical_price(S_base, K_base, T_base, r_base, true_sigma - h_sig)) / (2 * h_sig)

price_errors, delta_errors, gamma_errors = [], [], []
first_order_approx = []

for sv in sigma_hat:
    if sv <= 0:
        price_errors.append(np.nan); delta_errors.append(np.nan); gamma_errors.append(np.nan)
        first_order_approx.append(np.nan)
        continue
    p = bs.analytical_price(S_base, K_base, T_base, r_base, sv)
    d = bs.analytical_delta(S_base, K_base, T_base, r_base, sv)
    g = bs.analytical_gamma(S_base, K_base, T_base, r_base, sv)
    price_errors.append(p - true_price)
    delta_errors.append(d - true_delta_v)
    gamma_errors.append(g - true_gamma_v)
    first_order_approx.append(vega_numerical * (sv - true_sigma))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_data = [
    ('Option Price Error ($)', price_errors, first_order_approx, 'teal'),
    ('Delta Error', delta_errors, None, 'purple'),
    ('Gamma Error', gamma_errors, None, 'crimson'),
]

for ax, (title, errs, approx, color) in zip(axes, plot_data):
    ax.plot(epsilon_vol * 100, errs, color=color, lw=2, label='Exact Error')
    if approx is not None:
        ax.plot(epsilon_vol * 100, approx, 'k--', lw=1.5, alpha=0.7,
                label='1st-Order Taylor Approx')
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.axvline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_xlabel('Vol Estimation Error ε_σ (%)', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title}\nvs. Vol Estimation Error', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle(
    f'Error Propagation: How ε_σ Corrupts Greeks\n'
    f'(S={S_base}, K={K_base}, T=21d, σ_true={true_sigma:.0%})',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('vol_error_propagation.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"\n📌 A 1% vol error (ε_σ=0.01) creates:")
idx_1pct = np.argmin(np.abs(epsilon_vol - 0.01))
print(f"   Option Price Error: ${price_errors[idx_1pct]:.4f}")
print(f"   Delta Error: {delta_errors[idx_1pct]:.6f} (mishedge of {delta_errors[idx_1pct]*100:.4f} shares per $100 face)")
print(f"   Note: Price error ≈ Vega × ε_σ = {vega_numerical:.4f} × 0.01 = ${vega_numerical*0.01:.4f} (1st-order Taylor)")

---
## Section 3: Reference Strategies

Two baseline strategies are provided. Your task in Section 4 is to **fine-tune** their hyperparameters.

### Strategy 1: Delta-Neutral Scalping

**Theoretical Basis:** A delta-neutral portfolio earns the difference between **realized variance** and **implied variance** (the variance risk premium). The P&L of a continuously delta-hedged option is:

$$dP\!\&\!L = \frac{1}{2} \Gamma S^2 (\sigma_{\text{realized}}^2 - \sigma_{\text{implied}}^2) dt$$

**Error Impact:** If you compute $\Gamma$ numerically with the wrong step $h$, or if $\sigma_{\text{implied}}$ is mis-estimated, this P&L formula gives the wrong signal. You are literally trading on a miscalculated number.

### Strategy 2: Volatility Arbitrage

**Theoretical Basis:** When `volatility` (implied) significantly deviates from a rolling historical estimate (proxy for realized), the option is mispriced. Enter when the z-score exceeds a threshold:

$$z_t = \frac{\sigma_{\text{IV},t} - \mu_{\text{HV}}}{\sigma_{\text{HV}}}$$

**Error Impact:** The rolling window length $W$ introduces a lag bias. A window that's too short is noisy; too long misses regime changes.

In [ ]:
# ============================================================
# SECTION 3A: HYPERPARAMETER CONFIGURATION
# ⚡ STUDENT TASK: Tune these in Section 4 (80-Minute Challenge)
# ============================================================

class StrategyConfig:
    """
    Central hyperparameter block. Modify these values to tune strategy behavior.
    All parameters are annotated with their expected ranges and effects.
    """
    
    # -----------------------------------------------
    # STRATEGY 1: DELTA-NEUTRAL SCALPING
    # -----------------------------------------------
    
    # How often to rebalance the delta hedge (in bars; 1 bar = 5 sec)
    # LOWER = more precise hedge, HIGHER = fewer transaction costs
    # Range: [1, 60]  |  Default: 12 (every 60 seconds)
    hedging_frequency: int = 12
    
    # Delta tolerance before hedging: only rebalance if |delta| > threshold
    # Range: [0.01, 0.20]  |  Default: 0.05
    delta_tolerance: float = 0.05
    
    # Numerical step size for Greek computation (as fraction of spot)
    # Range: [0.001, 0.05]  |  Default: 0.01
    greek_h_fraction: float = 0.01
    
    # Use analytical (QuantLib) or numerical Greeks for hedging
    # True = analytical (more accurate), False = numerical (faster)
    use_analytical_greeks: bool = True
    
    # Number of option contracts per trade
    # Range: [1, 10]  |  Default: 1
    scalp_contracts: int = 1
    
    # -----------------------------------------------
    # STRATEGY 2: VOLATILITY ARBITRAGE
    # -----------------------------------------------
    
    # Z-score threshold to enter a vol-arb trade
    # HIGHER = fewer, higher-conviction trades; LOWER = more trades, more noise
    # Range: [1.0, 3.5]  |  Default: 2.0
    entry_z_score: float = 2.0
    
    # Z-score threshold to EXIT a vol-arb position
    # Range: [0.1, 1.0]  |  Default: 0.5
    exit_z_score: float = 0.5
    
    # Rolling window for historical vol estimation (in bars)
    # Range: [20, 500]  |  Default: 120 (10 minutes)
    vol_window: int = 120
    
    # Maximum concurrent vol-arb position size (contracts)
    # Range: [1, 5]  |  Default: 2
    max_vol_arb_contracts: int = 2
    
    # Minimum bid-ask spread quality filter (don't trade illiquid options)
    # Range: [0.01, 0.20]  |  Default: 0.05
    max_spread_filter: float = 0.05
    
    # -----------------------------------------------
    # RISK MANAGEMENT (BOTH STRATEGIES)
    # -----------------------------------------------
    
    # Stop-loss: exit if NAV falls below this fraction of initial NAV
    # Range: [0.80, 0.99]  |  Default: 0.95
    stop_loss_pct: float = 0.95
    
    # Don't trade in final N bars (avoid pin risk at expiry)
    # Range: [0, 720]  |  Default: 360 (30 minutes)
    no_trade_zone_bars: int = 360


# Instantiate with defaults
config = StrategyConfig()
print("Strategy Configuration:")
for attr in dir(config):
    if not attr.startswith('_'):
        print(f"  {attr:35s}: {getattr(config, attr)}")

In [ ]:
# ============================================================
# SECTION 3B: STRATEGY 1 — DELTA-NEUTRAL SCALPING
# Using greeks-package for hedge computation
# ============================================================

class DeltaNeutralScalper:
    """
    Delta-Neutral Scalping Strategy.
    
    Logic:
    ------
    1. Buy 1 ATM call at open.
    2. Every `hedging_frequency` bars, compute the portfolio delta.
    3. Hedge with underlying shares to maintain portfolio delta ≈ 0.
    4. Profit source: Gamma scalping (realized vol > implied vol).
    
    Error Sensitivity:
    ------------------
    - Wrong Greek h → wrong delta estimate → wrong hedge → residual delta → loss
    - Wrong sigma  → wrong BS price → wrong delta → wrong hedge
    """
    
    def __init__(self, bs_engine: BlackScholesEngine, cfg: StrategyConfig):
        self.bs = bs_engine
        self.cfg = cfg
        self.position_open = False
        self.bars_since_hedge = 0
        self.entry_bar = None
        self.trade_log = []
    
    def compute_hedge_delta(self, state: Dict) -> float:
        """Compute option delta using configured method (analytical vs. numerical)."""
        S  = state['uPrice']
        K  = state['strike']
        T  = state['time_to_expiry']
        r  = state['risk_free_rate']
        iv = state['volatility']
        ot = state['option_type']
        
        t0 = time.perf_counter()
        
        if self.cfg.use_analytical_greeks:
            if GREEKS_AVAILABLE:
                # greeks-package API
                try:
                    delta = greeks.delta(S, K, T, r, iv, ot[0].upper())
                except:
                    delta = self.bs.analytical_delta(S, K, T, r, iv, ot)
            else:
                delta = self.bs.analytical_delta(S, K, T, r, iv, ot)
        else:
            h = S * self.cfg.greek_h_fraction
            delta = self.bs.numerical_delta(S, K, T, r, iv, ot, h=h)
        
        return delta, time.perf_counter() - t0
    
    def on_bar(self, state: Dict, streamer: NYSE_Streamer) -> Optional[Dict]:
        """Called once per bar. Returns a trade dict if an order was submitted."""
        
        # Don't trade in no-trade zone
        if state['bars_remaining'] < self.cfg.no_trade_zone_bars:
            return None
        
        # Check bid-ask spread quality
        spread = state['0_ask'] - state['0_bid']
        if spread > self.cfg.max_spread_filter:
            return None
        
        t_decision_start = time.perf_counter()
        
        # ---- ENTRY: Buy 1 call at the start ----
        if not self.position_open and state['bar_index'] == 1:
            order = streamer.submit_order('buy', self.cfg.scalp_contracts, asset='option')
            if order.status == 'filled':
                self.position_open = True
                self.entry_bar = state['bar_index']
                self.trade_log.append({'event': 'entry', 'bar': state['bar_index'],
                                       'price': order.fill_price})
            return {'action': 'entry', 'order': order}
        
        if not self.position_open:
            return None
        
        # ---- DELTA HEDGE ----
        self.bars_since_hedge += 1
        if self.bars_since_hedge >= self.cfg.hedging_frequency:
            self.bars_since_hedge = 0
            
            delta_per_contract, dt = self.compute_hedge_delta(state)
            
            # Portfolio delta = option delta * contracts * multiplier + stock shares
            portfolio_delta = (delta_per_contract * 
                               streamer.option_position * 
                               streamer.multiplier) + streamer.stock_position
            
            if abs(portfolio_delta) > self.cfg.delta_tolerance * streamer.multiplier:
                # Hedge: sell/buy shares to bring delta to zero
                shares_needed = -int(round(portfolio_delta))
                if shares_needed != 0:
                    side = 'buy' if shares_needed > 0 else 'sell'
                    qty  = abs(shares_needed)
                    order = streamer.submit_order(
                        side, qty, asset='underlying',
                        decision_time_sec=time.perf_counter() - t_decision_start
                    )
                    return {'action': 'hedge', 'delta': portfolio_delta,
                            'shares': shares_needed, 'order': order, 'greek_time_ms': dt*1000}
        
        return None


print("✅ DeltaNeutralScalper class defined.")
print("   Key method: on_bar(state, streamer) → call every bar")

In [ ]:
# ============================================================
# SECTION 3C: STRATEGY 2 — VOLATILITY ARBITRAGE
# Using OptionStratLib for vol surface analysis
# ============================================================

class VolatilityArbitrage:
    """
    Volatility Arbitrage Strategy.
    
    Logic:
    ------
    1. Maintain a rolling window of implied vol (IV) and compute
       historical vol (HV) from underlying returns.
    2. When IV z-score is high (IV >> HV): SELL vol (sell option)
    3. When IV z-score is low  (IV << HV): BUY  vol (buy option)
    4. Exit when z-score reverts to exit_z_score.
    
    Error Sensitivity:
    ------------------
    - Short rolling window: high variance in HV estimate → false signals
    - Long  rolling window: stale HV, misses regime changes
    - The 'right' window is unknowable in advance (source of model error)
    """
    
    def __init__(self, cfg: StrategyConfig):
        self.cfg = cfg
        self.iv_history  = deque(maxlen=cfg.vol_window)
        self.hv_history  = deque(maxlen=cfg.vol_window)
        self.price_history = deque(maxlen=cfg.vol_window + 1)
        self.position    = 0     # +N long, -N short option contracts
        self.entry_z     = None
        self.trade_log   = []
    
    def _compute_hv(self) -> Optional[float]:
        """Compute annualized historical volatility from recent returns."""
        if len(self.price_history) < 10:
            return None
        prices = np.array(list(self.price_history))
        log_returns = np.diff(np.log(prices))
        # Annualize: bars are 5 seconds, 252*6.5*3600/5 bars/year
        bars_per_year = 252 * 6.5 * 3600 / 5
        return log_returns.std() * np.sqrt(bars_per_year)
    
    def _compute_zscore(self) -> Optional[float]:
        """Z-score of IV relative to recent IV history."""
        if len(self.iv_history) < 20:
            return None
        iv_arr = np.array(list(self.iv_history))
        mu, std = iv_arr.mean(), iv_arr.std()
        if std < 1e-8:
            return 0.0
        return (iv_arr[-1] - mu) / std
    
    def _compute_iv_hv_spread(self) -> Optional[float]:
        """Return IV - HV spread (the 'vol risk premium')."""
        hv = self._compute_hv()
        if hv is None or not self.iv_history:
            return None
        return list(self.iv_history)[-1] - hv
    
    def on_bar(self, state: Dict, streamer: NYSE_Streamer) -> Optional[Dict]:
        """Update history and generate vol-arb signals."""
        
        # Don't trade in no-trade zone
        if state['bars_remaining'] < self.cfg.no_trade_zone_bars:
            if self.position != 0:
                self._close_position(state, streamer, 'expiry_close')
            return None
        
        # Update rolling histories
        self.iv_history.append(state['volatility'])
        self.price_history.append(state['uPrice'])
        
        t_start = time.perf_counter()
        
        # Spread quality filter
        spread = state['0_ask'] - state['0_bid']
        if spread > self.cfg.max_spread_filter:
            return None
        
        z_score = self._compute_zscore()
        iv_hv   = self._compute_iv_hv_spread()
        
        if z_score is None:
            return None
        
        dt = time.perf_counter() - t_start
        result = {'z_score': round(z_score, 4), 'iv_hv': round(iv_hv or 0, 4),
                  'position': self.position, 'decision_ms': dt * 1000}
        
        # ---- EXIT LOGIC ----
        if self.position != 0 and abs(z_score) < self.cfg.exit_z_score:
            self._close_position(state, streamer, 'z_mean_revert')
            result['action'] = 'exit'
            return result
        
        # ---- ENTRY LOGIC ----
        max_pos = self.cfg.max_vol_arb_contracts
        
        if z_score > self.cfg.entry_z_score and self.position > -max_pos:
            # IV high relative to history → SELL vol (sell option, buy when it drops)
            order = streamer.submit_order(
                'sell', 1, asset='option',
                decision_time_sec=dt
            )
            if order.status == 'filled':
                self.position -= 1
                self.entry_z = z_score
                self.trade_log.append({'event': 'sell_vol', 'bar': state['bar_index'],
                                       'z': z_score, 'fill': order.fill_price})
            result['action'] = f'sell_vol (z={z_score:.2f})'
        
        elif z_score < -self.cfg.entry_z_score and self.position < max_pos:
            # IV low relative to history → BUY vol
            order = streamer.submit_order(
                'buy', 1, asset='option',
                decision_time_sec=dt
            )
            if order.status == 'filled':
                self.position += 1
                self.entry_z = z_score
                self.trade_log.append({'event': 'buy_vol', 'bar': state['bar_index'],
                                       'z': z_score, 'fill': order.fill_price})
            result['action'] = f'buy_vol (z={z_score:.2f})'
        
        return result
    
    def _close_position(self, state: Dict, streamer: NYSE_Streamer, reason: str):
        if self.position > 0:
            order = streamer.submit_order('sell', abs(self.position), asset='option')
        elif self.position < 0:
            order = streamer.submit_order('buy', abs(self.position), asset='option')
        else:
            return
        if order.status == 'filled':
            self.trade_log.append({'event': f'close_{reason}', 'bar': state['bar_index'],
                                   'fill': order.fill_price})
            self.position = 0


print("✅ VolatilityArbitrage class defined.")
print("   Key method: on_bar(state, streamer) → call every bar")

---
## Section 4: The 80-Minute Challenge 🏁

### Your Mission

You have **80 minutes** to:

1. **Run the baseline strategies** (cells below) and observe their default performance.
2. **Analyze the error plots** from Section 2 to understand WHY they perform the way they do.
3. **Tune the hyperparameters** in `StrategyConfig` (Section 3A) based on your analysis.
4. **Re-run** and compare your tuned version against the baseline.

### Scoring

| Metric | Weight | Description |
|--------|--------|-------------|
| Total PnL | 40% | Absolute profit in dollars |
| Sharpe Ratio | 35% | Risk-adjusted return |
| Computational Efficiency | 25% | Speed of decision-making |

### Key Questions to Answer

1. How does changing `greek_h_fraction` affect hedging quality and PnL?
2. What is the optimal `vol_window` for the z-score calculation? (Hint: think about the tradeoff between bias and variance in the HV estimator.)
3. Does using `use_analytical_greeks=True` vs `False` make a measurable difference in Sharpe? In efficiency?
4. At what `no_trade_zone_bars` do you successfully avoid pin risk losses?

In [ ]:
# ============================================================
# ⚡ STUDENT CELL: MODIFY YOUR HYPERPARAMETERS HERE
# ============================================================
# Copy StrategyConfig() below and change values.
# Then run Section 4 cells to evaluate your configuration.

# --- BASELINE CONFIG (do not modify this) ---
baseline_config = StrategyConfig()

# --- YOUR TUNED CONFIG (modify freely!) ---
my_config = StrategyConfig()

# STRATEGY 1 TUNING:
my_config.hedging_frequency      = 12       # Try: 1, 6, 12, 36, 120
my_config.delta_tolerance        = 0.05     # Try: 0.01, 0.05, 0.10, 0.20
my_config.greek_h_fraction       = 0.01     # Try: 0.001, 0.005, 0.01, 0.05
my_config.use_analytical_greeks  = True     # Try: True, False
my_config.scalp_contracts        = 1        # Try: 1, 2, 3

# STRATEGY 2 TUNING:
my_config.entry_z_score          = 2.0      # Try: 1.0, 1.5, 2.0, 2.5, 3.0
my_config.exit_z_score           = 0.5      # Try: 0.1, 0.3, 0.5, 1.0
my_config.vol_window             = 120      # Try: 20, 60, 120, 300, 500
my_config.max_vol_arb_contracts  = 2        # Try: 1, 2, 3, 5
my_config.max_spread_filter      = 0.05     # Try: 0.02, 0.05, 0.10, 0.20

# RISK MANAGEMENT:
my_config.stop_loss_pct          = 0.95     # Try: 0.90, 0.95, 0.98
my_config.no_trade_zone_bars     = 360      # Try: 0, 180, 360, 720

print("✅ Configurations ready. Proceed to run the backtest.")
print(f"\nBaseline: entry_z={baseline_config.entry_z_score}, "
      f"vol_window={baseline_config.vol_window}, h={baseline_config.greek_h_fraction}")
print(f"My Config: entry_z={my_config.entry_z_score}, "
      f"vol_window={my_config.vol_window}, h={my_config.greek_h_fraction}")

In [ ]:
# ============================================================
# SECTION 4B: THE BACKTEST ENGINE
# ============================================================

def run_backtest(
    data: pd.DataFrame,
    cfg: StrategyConfig,
    strategy_name: str = 'Vol Arb',
    initial_cash: float = 100_000.0,
    verbose: bool = False,
    max_bars: int = None,
) -> Dict:
    """
    Run a single backtest pass through the dataset.
    
    Both strategies run simultaneously on the same data.
    Returns full performance metrics and history.
    """
    streamer = NYSE_Streamer(data, initial_cash=initial_cash)
    bs_engine = BlackScholesEngine(use_quantlib=QUANTLIB_AVAILABLE)
    
    strategy1 = DeltaNeutralScalper(bs_engine, cfg)
    strategy2 = VolatilityArbitrage(cfg)
    
    nav_history = []
    action_log  = []
    z_log       = []
    
    n_bars = max_bars or len(data)
    
    print(f"\n{'='*55}")
    print(f"  BACKTEST: {strategy_name}")
    print(f"  Bars: {n_bars} | Initial NAV: ${initial_cash:,.0f}")
    print(f"{'='*55}")
    
    t_total_start = time.perf_counter()
    
    for i in range(n_bars):
        try:
            state = streamer.step()
        except StopIteration:
            break
        
        # Stop-loss check
        if state['nav'] < initial_cash * cfg.stop_loss_pct:
            print(f"  ⚠ STOP-LOSS triggered at bar {i} | NAV: ${state['nav']:,.2f}")
            break
        
        # Run strategies
        r1 = strategy1.on_bar(state, streamer)
        r2 = strategy2.on_bar(state, streamer)
        
        nav_history.append({'bar': i, 'timestamp': state['timestamp'],
                            'nav': state['nav'], 'uPrice': state['uPrice']})
        
        if r2 and 'z_score' in r2:
            z_log.append({'bar': i, 'z_score': r2['z_score'], 'iv_hv': r2.get('iv_hv', 0)})
        
        if r1 and 'action' in r1:
            action_log.append({'bar': i, 'strategy': 'scalper', **r1})
        if r2 and 'action' in r2:
            action_log.append({'bar': i, 'strategy': 'vol_arb', **r2})
        
        if verbose and i % 500 == 0:
            print(f"  Bar {i:5d} | NAV: ${state['nav']:>10,.2f} | "
                  f"S: {state['uPrice']:.2f} | "
                  f"IV: {state['volatility']:.1%} | "
                  f"Opt Pos: {streamer.option_position:+d}")
    
    t_total = time.perf_counter() - t_total_start
    
    metrics = streamer.get_performance_metrics()
    metrics['Wall Clock Time (s)'] = round(t_total, 2)
    metrics['Bars/Second'] = round(i / t_total, 0)
    
    print(f"\n  Results:")
    for k, v in metrics.items():
        print(f"    {k:35s}: {v}")
    
    return {
        'metrics': metrics,
        'nav_df': pd.DataFrame(nav_history).set_index('timestamp'),
        'z_df': pd.DataFrame(z_log) if z_log else None,
        'actions': action_log,
        'streamer': streamer,
    }


print("✅ Backtest engine ready. Run the next cell to execute.")

In [ ]:
# ============================================================
# SECTION 4C: RUN BASELINE vs. YOUR STRATEGY
# This is the core 80-minute challenge cell.
# ============================================================

# Run baseline
baseline_result = run_backtest(
    df_market, baseline_config,
    strategy_name='BASELINE',
    verbose=True,
    max_bars=4680,
)

print("\n" + "="*55)

# Run student's tuned strategy
my_result = run_backtest(
    df_market, my_config,
    strategy_name='MY TUNED STRATEGY',
    verbose=True,
    max_bars=4680,
)

In [ ]:
# ============================================================
# SECTION 4D: PERFORMANCE VISUALIZATION & COMPARISON
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('80-Minute Challenge: Performance Dashboard\nBaseline vs. My Tuned Strategy',
             fontsize=14, fontweight='bold')

# ---- Plot 1: NAV History ----
ax = axes[0, 0]
b_nav = baseline_result['nav_df']['nav']
m_nav = my_result['nav_df']['nav']
ax.plot(b_nav.values, label='Baseline', color='steelblue', lw=1.5, alpha=0.85)
ax.plot(m_nav.values, label='My Strategy', color='darkorange', lw=1.5, alpha=0.85)
ax.axhline(100_000, color='gray', linestyle='--', lw=1, label='Initial NAV')
ax.fill_between(range(len(b_nav)), 100_000, b_nav.values,
                alpha=0.1, color='steelblue')
ax.fill_between(range(len(m_nav)), 100_000, m_nav.values,
                alpha=0.1, color='darkorange')
ax.set_xlabel('Bar (5-second intervals)')
ax.set_ylabel('Portfolio NAV ($)')
ax.set_title('Portfolio NAV Over Time')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# ---- Plot 2: Drawdown ----
ax = axes[0, 1]
for nav_s, label, color in [(b_nav, 'Baseline', 'steelblue'), (m_nav, 'My Strategy', 'darkorange')]:
    navs = nav_s.values
    peak = np.maximum.accumulate(navs)
    dd = (navs - peak) / peak * 100
    ax.plot(dd, label=f'{label} (max {dd.min():.1f}%)', color=color, lw=1.5)
    ax.fill_between(range(len(dd)), 0, dd, alpha=0.15, color=color)
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('Bar')
ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdown Profile')
ax.legend()

# ---- Plot 3: IV z-score over time ----
ax = axes[1, 0]
if my_result['z_df'] is not None and len(my_result['z_df']) > 0:
    z_df = my_result['z_df']
    ax.plot(z_df['bar'], z_df['z_score'], color='purple', lw=1, alpha=0.8, label='IV z-score')
    ax.axhline(my_config.entry_z_score, color='red', linestyle='--', lw=1.5,
               label=f'Entry (+{my_config.entry_z_score})')
    ax.axhline(-my_config.entry_z_score, color='green', linestyle='--', lw=1.5,
               label=f'Entry (−{my_config.entry_z_score})')
    ax.axhline(my_config.exit_z_score, color='orange', linestyle=':', lw=1,
               label=f'Exit (±{my_config.exit_z_score})')
    ax.axhline(-my_config.exit_z_score, color='orange', linestyle=':', lw=1)
    ax.fill_between(z_df['bar'],
                    my_config.entry_z_score, z_df['z_score'].clip(lower=my_config.entry_z_score),
                    alpha=0.2, color='red', label='Sell vol signal')
    ax.fill_between(z_df['bar'],
                    -my_config.entry_z_score, z_df['z_score'].clip(upper=-my_config.entry_z_score),
                    alpha=0.2, color='green', label='Buy vol signal')
ax.set_xlabel('Bar')
ax.set_ylabel('IV Z-Score')
ax.set_title(f'Volatility Z-Score (window={my_config.vol_window} bars)')
ax.legend(fontsize=8)

# ---- Plot 4: Metric Comparison Bar Chart ----
ax = axes[1, 1]
b_m = baseline_result['metrics']
s_m = my_result['metrics']

compare_keys = ['Total PnL ($)', 'Sharpe Ratio (Ann.)', 'Max Drawdown (%)',
                'Computational Efficiency Score']
x = np.arange(len(compare_keys))
w = 0.35

b_vals = [b_m.get(k, 0) for k in compare_keys]
s_vals = [s_m.get(k, 0) for k in compare_keys]

bars1 = ax.bar(x - w/2, b_vals, w, label='Baseline', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, s_vals, w, label='My Strategy', color='darkorange', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([k.replace(' (', '\n(') for k in compare_keys], fontsize=9)
ax.set_title('Metric Comparison')
ax.legend()
ax.axhline(0, color='gray', lw=0.8)

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.1f}',
            ha='center', va='bottom' if h >= 0 else 'top', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.1f}',
            ha='center', va='bottom' if h >= 0 else 'top', fontsize=8)

plt.tight_layout()
plt.savefig('challenge_results.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SECTION 4E: FINAL SCORECARD & SENSITIVITY ANALYSIS
# ============================================================

def compute_composite_score(metrics: Dict, weights=None) -> float:
    """Compute a weighted composite score for the leaderboard."""
    if weights is None:
        weights = {'pnl': 0.40, 'sharpe': 0.35, 'efficiency': 0.25}
    
    pnl_score   = np.clip(metrics.get('Total PnL ($)', 0) / 1000, -100, 100)
    sharpe_score = np.clip(metrics.get('Sharpe Ratio (Ann.)', 0) * 20, -100, 100)
    eff_score   = metrics.get('Computational Efficiency Score', 50)
    
    return (weights['pnl'] * pnl_score + 
            weights['sharpe'] * sharpe_score + 
            weights['efficiency'] * eff_score)


b_score = compute_composite_score(baseline_result['metrics'])
m_score = compute_composite_score(my_result['metrics'])
improvement = m_score - b_score

print("\n" + "="*60)
print("  📊 FINAL SCORECARD")
print("="*60)
print(f"\n  {'Metric':40s} {'Baseline':>10s} {'My Strat':>10s} {'Δ':>8s}")
print("-" * 70)

keys_to_compare = [
    'Total PnL ($)', 'Return (%)', 'Sharpe Ratio (Ann.)',
    'Max Drawdown (%)', 'N Trades', 'N Rejected',
    'Avg Decision Time (ms)', 'Computational Efficiency Score',
]

for k in keys_to_compare:
    bv = baseline_result['metrics'].get(k, 'N/A')
    mv = my_result['metrics'].get(k, 'N/A')
    try:
        delta = mv - bv
        sign = '+' if delta >= 0 else ''
        print(f"  {k:40s} {bv:>10.3f} {mv:>10.3f} {sign}{delta:>7.3f}")
    except:
        print(f"  {k:40s} {str(bv):>10s} {str(mv):>10s}")

print("-" * 70)
print(f"  {'COMPOSITE SCORE':40s} {b_score:>10.2f} {m_score:>10.2f} {'+' if improvement>=0 else ''}{improvement:>7.2f}")
print("="*60)

if improvement > 5:
    print(f"\n  🏆 Excellent! Your strategy improved by {improvement:.1f} points.")
elif improvement > 0:
    print(f"\n  ✅ Good work! Marginal improvement of {improvement:.1f} points.")
elif improvement > -5:
    print(f"\n  📉 Slightly underperformed baseline ({improvement:.1f} pts). Keep tuning!")
else:
    print(f"\n  ⚠ Strategy degraded significantly ({improvement:.1f} pts). Review your config.")

# --- Sensitivity Table: entry_z_score sweep ---
print("\n" + "="*60)
print("  📈 SENSITIVITY ANALYSIS: entry_z_score")
print("="*60)
print(f"\n  {'entry_z':>10s} {'PnL':>10s} {'Sharpe':>10s} {'N Trades':>10s} {'Score':>8s}")
print("-" * 52)

for z_test in [1.0, 1.5, 2.0, 2.5, 3.0]:
    cfg_test = StrategyConfig()
    cfg_test.entry_z_score = z_test
    cfg_test.vol_window = my_config.vol_window
    result_test = run_backtest(df_market, cfg_test,
                               strategy_name=f'z={z_test}', verbose=False)
    m = result_test['metrics']
    sc = compute_composite_score(m)
    print(f"  {z_test:>10.1f} {m.get('Total PnL ($)', 0):>10.2f} "
          f"{m.get('Sharpe Ratio (Ann.)', 0):>10.4f} "
          f"{m.get('N Trades', 0):>10d} {sc:>8.2f}")

---
## Section 5: Post-Lab Reflection Questions

Answer these in the cell below using Markdown. These are graded.

---

### Q1 — Error Decomposition
From Section 2B, identify the optimal step size $h^*$ for Delta and Gamma numerically. Now derive it analytically for the central-difference formula. Where do your numerical and analytical results agree? Where do they diverge?

### Q2 — The Pin Risk Problem
Using the error surface from Section 2C, describe in your own words why a market-maker running a delta-hedged book in single-name equity options faces catastrophic risk in the last 30 minutes of trading on expiration Friday. What real-world practice do traders use to mitigate this?

### Q3 — Taylor Series and PnL
The P&L of a delta-hedged option position over time interval $\delta t$ is approximately:
$$\text{PnL} = \frac{1}{2}\Gamma(\delta S)^2 - \Theta \cdot \delta t$$
This is the **Gamma-Theta tradeoff**. Derived from the Taylor expansion, it shows that buying options profits when $\Gamma (\delta S)^2 > \Theta \delta t$, i.e., when realized moves are large enough to pay for time decay. Using your backtest results, identify periods where this condition held and periods where it did not.

### Q4 — Computational Efficiency
Your `Computational Efficiency Score` penalizes slow decisions. In HFT (High-Frequency Trading), decisions must be made in microseconds ($10^{-6}$ seconds). Assuming QuantLib's `analytical_delta` takes ~1ms per evaluation:
- How many delta computations per second can you perform?
- What numerical approximation or pre-computation trick could you use to achieve 100× speedup?
- Is there a fundamental accuracy-speed tradeoff? Express it mathematically.

### Q5 — Your Best Configuration
Report your final hyperparameter configuration and explain, using the mathematical framework from Section 2, why each parameter choice improves performance. Be specific about which error source each choice addresses.

### 📝 YOUR ANSWERS

**Q1:** *(Double-click to edit)*

---

**Q2:**

---

**Q3:**

---

**Q4:**

---

**Q5:**


In [1]:
# ============================================================
# BONUS: GREEKS OVER TIME — Visualize how all Greeks evolve
# as the option approaches expiration
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()

greek_cols = ['0_delta', '0_gamma', '0_vega', '0_theta', '0_rho', 'volatility']
greek_labels = ['Delta (Δ)', 'Gamma (Γ)', 'Vega (ν)', 'Theta (Θ)', 'Rho (ρ)', 'Implied Vol (σ)']
greek_colors = ['royalblue', 'tomato', 'mediumseagreen', 'purple', 'goldenrod', 'teal']

for ax, col, label, color in zip(axes, greek_cols, greek_labels, greek_colors):
    y = df_market[col].values
    x = np.arange(len(y))
    ax.plot(x, y, color=color, lw=1.2, alpha=0.85)
    ax.fill_between(x, y.min(), y, alpha=0.10, color=color)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Bar (5-sec)', fontsize=9)
    ax.tick_params(labelsize=8)
    
    # Annotate mean and std
    ax.text(0.98, 0.95, f'μ={y.mean():.4f}\nσ={y.std():.4f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('All Greeks Over a Full Trading Day\n'
             '(5-second intervals, stochastic volatility + GBM underlying)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('greeks_over_time.png', dpi=130, bbox_inches='tight')
plt.show()

print("\n✅ Lab complete!")
print("="*55)
print("  Files generated:")
print("    error_curves.png         — Step size vs. error")
print("    pin_risk_surface.png     — 3D error surface")
print("    vol_error_propagation.png— Vol error → Greek errors")
print("    challenge_results.png    — Your strategy vs. baseline")
print("    greeks_over_time.png     — All Greeks time-series")
print("="*55)

NameError: name 'plt' is not defined